# 🚀 AI 安全模型训练 - Qwen2.5-7B

**公司**: 杭州市上城区乐友信息服务工作室

---

## 第一步: 连接 GPU

In [ ]:
# 验证 GPU 连接
!nvidia-smi

In [ ]:
# 检查 PyTorch 和 CUDA
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"显存: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

---

## 第二步: 克隆代码并安装依赖

In [ ]:
# 克隆 GitHub 仓库
!git clone https://github.com/richard3153/nexify-landing.git
cd nexify-landing
!ls

In [ ]:
# 安装训练依赖
!pip install -q transformers accelerate peft bitsandbytes
!pip install -q datasets loralib huggingface_hub
!pip install -q scipy

---

## 第三步: 下载训练数据

In [ ]:
# 创建数据目录
!mkdir -p nexify-landing/data

# 下载 BeaverTails 安全对话数据集
from datasets import load_dataset

print("📥 下载 BeaverTails 数据集...")
beavertails = load_dataset("PKU-Alignment/BeaverTails", split="train[:5000]")
print(f"✅ 下载完成: {len(beavertails)} 条样本")

# 保存为 JSONL
import json

with open("nexify-landing/data/train.json", "w", encoding="utf-8") as f:
    for i, item in enumerate(beavertails):
        record = {
            "id": i,
            "instruction": item["prompt"],
            "output": item["response"]
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"💾 保存到 nexify-landing/data/train.json")

---

## 第四步: 加载模型 (4bit 量化)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# 4bit 量化配置
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print("🔄 加载 Qwen2.5-7B 模型...")
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    padding_side='right'
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map="auto",
)

print("✅ 模型加载完成!")

---

## 第五步: 配置 LoRA

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

# 准备模型进行训练
model = prepare_model_for_kbit_training(model)

# LoRA 配置
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    bias="none",
)

# 应用 LoRA
model = get_peft_model(model, lora_config)

# 显示可训练参数
model.print_trainable_parameters()

---

## 第六步: 准备训练数据

In [ ]:
from datasets import load_dataset

# 加载训练数据
def format_example(example):
    text = f"### 指令:\n{example['instruction']}\n\n### 回答:\n{example['output']}"
    return {"text": text}

dataset = load_dataset("json", data_files="nexify-landing/data/train.json", split="train")
dataset = dataset.map(format_example, remove_columns=dataset.column_names)

print(f"📊 训练数据: {len(dataset)} 条")
print(f"示例: {dataset[0]['text'][:200]}...")

In [ ]:
MAX_LENGTH = 512

def tokenize(example):
    inputs = tokenizer(
        example["text"],
        max_length=MAX_LENGTH,
        truncation=True,
        padding="max_length"
    )
    inputs["labels"] = inputs["input_ids"].copy()
    return inputs

tokenized_dataset = dataset.map(
    tokenize,
    batched=True,
    remove_columns=["text"]
)

print(f"✅ Tokenize 完成: {len(tokenized_dataset)} 条")

---

## 第七步: 开始训练

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

# 训练参数
training_args = TrainingArguments(
    output_dir="./output",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    warmup_steps=100,
    report_to="none",
    fp16=False,
)

# 数据整理器
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
)

# 训练器
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

# 开始训练
print("🚀 开始训练...")
trainer.train()

---

## 第八步: 保存模型

In [ ]:
# 保存 LoRA 权重
model.save_pretrained("./output/lora_adapter")
tokenizer.save_pretrained("./output/lora_adapter")

print("✅ 模型保存完成!")
print("📁 保存路径: ./output/lora_adapter")

In [ ]:
# 下载模型到本地
from google.colab import files
!zip -r lora_adapter.zip ./output/lora_adapter
files.download("lora_adapter.zip")

print("📦 模型已打包下载")

---

## 第九步: 推送至 HuggingFace

In [ ]:
# 登录 HuggingFace
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# 创建 HuggingFace 仓库并上传
from huggingface_hub import HfApi, create_repo

# 配置你的 HuggingFace 用户名
HF_USERNAME = "你的HF用户名"  # 👈 修改这里
MODEL_NAME = "qwen2.5-7b-ai-safety"
REPO_NAME = f"{HF_USERNAME}/{MODEL_NAME}"

# 创建仓库
try:
    create_repo(REPO_NAME, exist_ok=True)
    print(f"✅ 仓库已创建: https://huggingface.co/{REPO_NAME}")
except Exception as e:
    print(f"⚠️ {e}")

# 上传模型
api = HfApi()
api.upload_folder(
    folder_path="./output/lora_adapter",
    repo_id=REPO_NAME,
    repo_type="model",
)

print(f"🎉 模型已发布: https://huggingface.co/{REPO_NAME}")